# EfficientNetB0: setup and one-batch check

Run this notebook before starting the long training run. It reuses your team's
CSV manifests, RGB conversion, augmentation and ImageNet normalization.
**This is an untrained starter: no waste-classification accuracy is claimed.**

Install dependencies using `docs/efficientnetb0_start_here.md`, select the
Waste Classification Python kernel, and obtain the exact shared image folder.
Resolve the team's perceptual-duplicate review before full experiments.

## 1. Load the project and select CUDA, Apple MPS, or CPU

In [ ]:
from pathlib import Path
import sys
import os
import json
import hashlib
import platform
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import yaml
import torch
import torchvision
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

# Works when opened from either the repository root or notebooks/.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'src/data/dataloader.py').is_file()
     and (p / 'configs/base.yaml').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook inside your cloned waste-classification repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataloader import create_dataloaders
from src.models.efficientnetb0 import create_efficientnetb0
from src.utils.seed import set_seed

with (PROJECT_ROOT / 'configs/base.yaml').open() as handle:
    base_config = yaml.safe_load(handle)
with (PROJECT_ROOT / 'configs/efficientnetb0.yaml').open() as handle:
    config = yaml.safe_load(handle)

set_seed(config['experiment']['seed'])
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Project:', PROJECT_ROOT)
print('Device:', device)
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('ImageNet weights will download automatically if not already cached.')

## 2. Verify the shared files and labels

Reading test file names here is a file-integrity check only. The model never
receives test images during development. Keep the original manifests unchanged.

In [ ]:
# Get the SAME image folder used by your teammate. Do not recreate the splits.
# You may change this path if the dataset is stored outside the repository.
DATASET_ROOT = Path(os.environ.get(
    'WASTE_DATASET_ROOT',
    str(PROJECT_ROOT / base_config['data']['dataset_root']),
)).expanduser()
MANIFEST_DIR = PROJECT_ROOT / base_config['data']['manifest_dir']

frames = {split: pd.read_csv(MANIFEST_DIR / f'{split}.csv')
          for split in ['train', 'val', 'test']}
expected_classes = base_config['classes']
assert config['model']['num_classes'] == len(expected_classes)
assert config['model']['freeze_backbone'] is True, 'This notebook is the frozen baseline.'
assert config['model']['pretrained'] is True, 'The baseline uses pretrained ImageNet weights.'
assert base_config['data']['image_size'] == 224, 'Shared transforms currently use 224 pixels.'

seen_paths = set()
for split, frame in frames.items():
    assert {'relative_path', 'class_name', 'split'} <= set(frame.columns)
    assert len(frame) > 0 and not frame.isna().any().any()
    assert frame['relative_path'].is_unique, f'Duplicate paths within {split}'
    assert frame['split'].eq(split).all(), f'Incorrect split labels in {split}'
    assert set(frame['class_name']) == set(expected_classes)
    paths = set(frame['relative_path'])
    assert not seen_paths.intersection(paths), f'Overlapping paths found in {split}'
    seen_paths.update(paths)
    assert all(Path(row.relative_path).parts[0] == row.class_name
               for row in frame.itertuples()), f'Folder/label mismatch in {split}'

summary = pd.DataFrame({'split': list(frames),
                        'images': [len(frame) for frame in frames.values()]})
display(summary)
print('CSV structure, class names and path disjointness passed.')
print('This does not verify near-duplicate content or the correctness of every label.')

# Check presence of files; no test image is loaded or used for model selection.
missing = [relative_path for relative_path in sorted(seen_paths)
           if not (DATASET_ROOT / relative_path).is_file()]
if missing:
    raise FileNotFoundError(
        f'{len(missing)} manifest images are missing under {DATASET_ROOT}. '
        f'Examples: {missing[:5]}. Obtain the exact shared dataset; do not rename '
        'files or regenerate the CSV splits to hide this error.'
    )
print('All manifest image paths exist.')

loaders = create_dataloaders(
    dataset_root=DATASET_ROOT,
    manifest_dir=MANIFEST_DIR,
    batch_size=config['training']['batch_size'],
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)
class_to_idx = loaders['class_to_idx']
class_names = loaders['train_dataset'].classes
assert class_names == expected_classes
train_loader = loaders['train_loader']
val_loader = loaders['val_loader']
print('Shared class mapping:', class_to_idx)

## 3. Create the pretrained model and check one real training batch

In [ ]:
model = create_efficientnetb0(**{
    key: config['model'][key]
    for key in ['num_classes', 'pretrained', 'freeze_backbone']
}).to(device)
model.eval()
images, labels = next(iter(train_loader))
with torch.inference_mode():
    logits = model(images.to(device))
    loss = torch.nn.CrossEntropyLoss()(logits, labels.to(device))

assert logits.shape == (images.shape[0], len(class_names))
assert torch.isfinite(logits).all().item() and torch.isfinite(loss).item()
print('Input:', tuple(images.shape))
print('Output:', tuple(logits.shape))
print('Initial batch loss:', float(loss.item()))
print('This loss is a wiring check, not a trained-model result.')
print('Total parameters:', sum(p.numel() for p in model.parameters()))
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Confirm that the frozen extractor stays frozen during training

In [ ]:
model.train()
assert not model.network.features.training
assert model.network.classifier.training
assert all(not p.requires_grad for p in model.network.features.parameters())
assert all(p.requires_grad for p in model.network.classifier.parameters())
print('Feature extractor frozen; classifier trainable; BatchNorm statistics held fixed.')
model.eval()

## 5. Save a small, real check record

Commit this record only after this notebook has actually passed on your setup.
It records a technical check, not training or evaluation accuracy.

In [ ]:
record_dir = PROJECT_ROOT / 'experiments/efficientnetb0'
record_dir.mkdir(parents=True, exist_ok=True)
record = {
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'check': 'one_real_training_batch_forward_pass',
    'device': str(device),
    'torch': str(torch.__version__),
    'torchvision': str(torchvision.__version__),
    'input_shape': list(images.shape),
    'output_shape': list(logits.shape),
    'finite_loss': True,
    'class_to_idx': class_to_idx,
    'manifest_sha256': {
        name: hashlib.sha256((MANIFEST_DIR / f'{name}.csv').read_bytes()).hexdigest()
        for name in ['train', 'val', 'test']
    },
}
record_path = record_dir / 'smoke_check.json'
record_path.write_text(json.dumps(record, indent=2) + '\n')
print('Saved:', record_path.relative_to(PROJECT_ROOT))

## Next step

Save this notebook, commit the successful check, then open
`07_efficientnetb0_frozen_training.ipynb`. If paths are missing, obtain the shared
dataset or change `DATASET_ROOT`; do not regenerate the team's splits.